## Query Tracker Test

Validates cross-language query ID capture and RESULT_SCAN reuse.
The query tracker hooks into `ServerConnection.add_query_listener()`
to capture every query ID executed through the shared Snowpark session.


In [ ]:
from sfnb_setup import setup_notebook
result = setup_notebook(config="query_tracker_test_config.yaml")

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Session ready: {session.get_current_warehouse()}")

## 1. Verify Tracker + Python Query Capture


In [ ]:
from query_tracker import nb_last_query_id, nb_query_id, get_registry

py_result = session.sql(
    "SELECT 'from_python' AS origin, 42 AS answer"
).collect()
print(f"Python query result: {py_result}")

py_qid = nb_last_query_id()
print(f"Captured query ID: {py_qid}")

if py_qid:
    verify = session.sql(
        f"SELECT * FROM TABLE(RESULT_SCAN('{py_qid}'))"
    ).to_pandas()
    assert verify.iloc[0]["ANSWER"] == 42
    print("[PASS] Python query capture + RESULT_SCAN")
else:
    print("[SKIP] Tracker not capturing in this environment")

## 2. Second Python Query


In [ ]:
session.sql("SELECT 'second' AS origin, 99 AS value").collect()
qid2 = nb_last_query_id()
print(f"Second query ID: {qid2}")

if qid2 and py_qid:
    assert qid2 != py_qid, "Second query should have distinct ID"
    verify2 = session.sql(
        f"SELECT * FROM TABLE(RESULT_SCAN('{qid2}'))"
    ).to_pandas()
    assert verify2.iloc[0]["VALUE"] == 99
    print("[PASS] Second query capture")
else:
    print("[SKIP] Tracker not capturing")

## 3. R -- Consume Python Result via sfr_result_scan()


In [ ]:
qid = py_qid if py_qid else "none"

In [ ]:
%%R -i qid
library(snowflakeR)
conn <- sfr_connect()

if (qid != "none") {
    df <- sfr_result_scan(conn, query_id = qid)
    cat("R received via RESULT_SCAN:\n")
    print(df)
    stopifnot(df$ANSWER == 42)
    cat("[PASS] R sfr_result_scan\n")
} else {
    df <- sfr_query(conn, "SELECT 1 AS test_col")
    print(df)
    cat("[SKIP] R sfr_result_scan (no qid)\n")
}

## 4. Summary + CI Results


In [ ]:
reg = get_registry()
n = len(reg['all'])
print(f"Queries captured: {n}")
for i, e in enumerate(reg['all']):
    print(f"  [{i}] {e['query_id'][:25]}... {(e.get('sql') or '')[:50]}")
print("\nAll tests completed.")

try:
    _ci = get_active_session()
    _ci.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.QUERY_TRACKER_TEST_RESULTS AS
        SELECT 'query_tracker_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written.")
except Exception:
    pass